# 00 — Data Quality

**Purpose:** Load raw MMEX data, run validation, inspect the quality report.
Save the validated DataFrame to `data/interim/transactions_validated.parquet`.

**Inputs:** `data/raw/*.mmb` or `data/raw/*.csv`  
**Outputs:** `data/interim/transactions_validated.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from IPython.display import display

from src.utils.logging_config import setup_notebook_logging
from src.ingestion.mmex_sqlite_parser import parse_mmex_sqlite
from src.ingestion.mmex_csv_parser import parse_mmex_csv
from src.ingestion.validator import validate_transactions
from src.storage.local_writer import LocalWriter

logger = setup_notebook_logging()

In [ ]:
# ── CONFIGURE: set the path to your MMEX file ─────────────────────────────────
MMB_PATH = "../data/raw/my_finances.mmb"   # ← edit this
# If you have a CSV export instead, comment the line above and use:
# CSV_PATH = "../data/raw/mmex_export_YYYY-MM-DD.csv"
# ──────────────────────────────────────────────────────────────────────────────

df = parse_mmex_sqlite(MMB_PATH)
# df = parse_mmex_csv(CSV_PATH)  # uncomment for CSV path

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

In [ ]:
# ── Step 2: Validate ──────────────────────────────────────────────────────────
report = validate_transactions(df)
report.display()

In [ ]:
# ── Step 3: Column summary ────────────────────────────────────────────────────
info_df = pd.DataFrame({
    "dtype":      df.dtypes,
    "non_null":   df.count(),
    "null_count": df.isna().sum(),
    "null_pct":   (df.isna().mean() * 100).round(2),
    "nunique":    df.nunique(),
})
display(info_df)

In [ ]:
# ── Step 4: Descriptive statistics ───────────────────────────────────────────
display(df.describe().T.round(2))

for col in df.select_dtypes(include=["datetime64[ns]"]).columns:
    print(f"\n{col}: {df[col].min()} → {df[col].max()} ({df[col].nunique()} unique dates)")

for col in df.select_dtypes(include=["object"]).columns:
    print(f"\n{col} — top 10 values:")
    display(df[col].value_counts().head(10))

In [ ]:
# ── Step 5: Spot-check rows ───────────────────────────────────────────────────
display(df.sample(min(10, len(df)), random_state=42))

In [ ]:
# ── Step 6: Known-issue checks ────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

if "date" in df.columns:
    date_series = df["date"].sort_values()
    gaps = date_series.diff()
    big_gaps = gaps[gaps > pd.Timedelta(days=14)]
    if len(big_gaps):
        print(f"\nDate gaps > 14 days ({len(big_gaps)} gap(s)):")
        for idx in big_gaps.index:
            gap_start = date_series.iloc[list(date_series.index).index(idx) - 1]
            gap_end   = date_series.loc[idx]
            n_days    = (gap_end - gap_start).days
            print(f"  {gap_start.date()} → {gap_end.date()} ({n_days} days)")
    else:
        print("No date gaps > 14 days.")

In [ ]:
# ── Step 7: Save — only if validation passed ─────────────────────────────────
assert report.is_passing(), "Validation failed — fix critical issues before saving"

writer = LocalWriter(project_root="..")
writer.save_interim(df, "transactions_validated")
print("Saved: data/interim/transactions_validated.parquet")